# 02 Generate FACT Data
This notebook generates transactional data (Invoices and Orders) linked to the DIM snapshots, strictly following KiotViet templates.

In [1]:
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta

# Scenario Configuration
CONFIG = {
    'date_range': ('2026-03-01', '2026-03-31'),
    'invoices_per_day': (10, 30),
    'orders_per_day': (3, 10),
    'basket_size': (1, 6),
    'seed': 42,
    'chunk_size': 1000 # KiotViet UI import limit
}

np.random.seed(CONFIG['seed'])

INVOICE_HEADERS = [
    'Mã hóa đơn', 'Thời gian', 'Người bán', 'Kênh bán', 'Mã khách hàng', 
    'Tên khách hàng', 'Điện thoại', 'Email', 'Địa chỉ (Khách hàng)', 
    'Khu vực (Khách hàng)', 'Phường/Xã (Khách hàng)', 'Bảng giá', 
    'Mã hàng', 'Số lượng', 'Đơn giá', 'Giảm giá %', 'Giảm giá', 
    'Giảm giá hóa đơn', 'Giảm giá hóa đơn %', 'Tiền mặt', 'Thẻ', 
    'Chuyển khoản', 'Ghi chú'
]

ORDER_HEADERS = [
    'Mã đặt hàng', 'Mã hóa đơn', 'Mã vận đơn', 'Địa chỉ lấy hàng', 'Thời gian', 'Thời gian tạo', 
    'Mã khách hàng', 'Tên khách hàng', 'Điện thoại', 'Địa chỉ (Khách hàng)', 
    'Khu vực (Khách hàng)', 'Phường/Xã (Khách hàng)', 'Người nhận đặt', 'Kênh bán', 
    'Người tạo', 'Đối tác giao hàng', 'Người nhận', 'Điện thoại (Người nhận)', 
    'Địa chỉ (Người nhận)', 'Khu vực (Người nhận)', 'Phường/Xã (Người nhận)', 
    'Dịch vụ', 'Trọng lượng (gram)', 'Dài', 'Rộng', 'Cao', 'Phí trả ĐTGH (trả ĐT)', 
    'Ghi chú', 'Tổng tiền hàng', 'Giảm giá phiếu đặt', 'Khách cần trả', 'Khách đã trả', 
    'Tiền mặt', 'Thẻ', 'Chuyển khoản', 'Ví', 'ĐVT', 'Điểm', 'Thời gian giao hàng', 'Trạng thái', 
    'Mã hàng', 'Mã vạch', 'Tên hàng', 'Thương hiệu', 'Ghi chú hàng hóa', 'Số lượng', 
    'Đơn giá', 'Giảm giá %', 'Giảm giá', 'Giá bán', 'Thành tiền'
]

print("FACT Generation Configured with exact KiotViet headers.")

FACT Generation Configured with exact KiotViet headers.


## 1. Load DIM Snapshots and Prepare Distributions

In [2]:
df_products = pd.read_csv('../data/out/csv/dim_products.csv')
df_customers = pd.read_csv('../data/out/csv/dim_customers.csv')
df_employees = pd.read_csv('../data/out/csv/dim_employees.csv')

# Realistic Product Distribution (Zipf's Law) - some items are bought way more
n_prods = len(df_products)
a = 1.5 # Zipf parameter
weights = 1 / (np.arange(1, n_prods + 1) ** a)
weights /= weights.sum()
# Assign these weights randomly to our products
np.random.shuffle(weights)
df_products['weight'] = weights

print(f"Syncing with DIMs: {len(df_products)} products, {len(df_customers)} customers.")

Syncing with DIMs: 100 products, 50 customers.


## 2. Generate Transaction Data (Invoices and Orders)

In [3]:
invoice_rows = []
order_rows = []
start_date = datetime.strptime(CONFIG['date_range'][0], '%Y-%m-%d')
end_date = datetime.strptime(CONFIG['date_range'][1], '%Y-%m-%d')
delta = end_date - start_date

invoice_id_counter = 1
order_id_counter = 1

for i in range(delta.days + 1):
    current_date = start_date + timedelta(days=i)
    
    # Generate Invoices
    daily_invoices = np.random.randint(*CONFIG['invoices_per_day'])
    for _ in range(daily_invoices):
        invoice_code = f"HDIP{invoice_id_counter:05d}"
        customer = df_customers.sample(1).iloc[0]
        seller = df_employees.sample(1).iloc[0]
        # Realistic time distribution (more sales in afternoon)
        hour = int(np.random.normal(15, 3))%24 if np.random.rand() > 0.3 else np.random.randint(8, 22)
        hour = max(8, min(hour, 22))
        time_str = (current_date + timedelta(hours=hour, minutes=np.random.randint(0, 59))).strftime('%Y-%m-%d %H:%M:%S')
        
        num_lines = np.random.randint(*CONFIG['basket_size'])
        prods = df_products.sample(n=num_lines, replace=False, weights='weight')
        
        invoice_total = 0
        lines = []
        for _, prod in prods.iterrows():
            qty = np.random.randint(1, 4)
            price = prod['Giá bán']
            invoice_total += (price * qty)
            lines.append((prod['Mã hàng'], qty, price))

        # Payment Logic
        pay_method = np.random.choice(['Tiền mặt', 'Chuyển khoản', 'Thẻ'], p=[0.5, 0.4, 0.1])

        for j, (p_code, p_qty, p_price) in enumerate(lines):
            row = {h: np.nan for h in INVOICE_HEADERS}
            row['Mã hóa đơn'] = invoice_code
            row['Thời gian'] = time_str
            row['Người bán'] = seller['Tên nhân viên (*)']
            row['Kênh bán'] = 'Bán trực tiếp'
            row['Mã khách hàng'] = customer['Mã khách hàng']
            row['Tên khách hàng'] = customer['Tên khách hàng']
            row['Điện thoại'] = customer['Điện thoại']
            row['Mã hàng'] = p_code
            row['Số lượng'] = p_qty
            row['Đơn giá'] = p_price
            if j == 0:
                row[pay_method] = invoice_total
            invoice_rows.append(row)
        invoice_id_counter += 1

    # Generate Orders
    daily_orders = np.random.randint(*CONFIG['orders_per_day'])
    for _ in range(daily_orders):
        order_code = f"DHIP{order_id_counter:05d}"
        customer = df_customers.sample(1).iloc[0]
        seller = df_employees.sample(1).iloc[0]
        hour = max(8, min(int(np.random.normal(12, 4))%24, 21))
        time_str = (current_date + timedelta(hours=hour, minutes=np.random.randint(0, 59))).strftime('%Y-%m-%d %H:%M:%S')
        
        num_lines = np.random.randint(*CONFIG['basket_size'])
        prods = df_products.sample(n=num_lines, replace=False, weights='weight')
        
        order_total = 0
        lines = []
        for _, prod in prods.iterrows():
            qty = np.random.randint(1, 4)
            price = prod['Giá bán']
            order_total += (price * qty)
            lines.append((prod['Mã hàng'], prod['Tên hàng'], qty, price))
            
        status = np.random.choice(['Hoàn thành', 'Phiếu tạm', 'Đang giao hàng'], p=[0.7, 0.1, 0.2])
        pay_method = np.random.choice(['Tiền mặt', 'Chuyển khoản'], p=[0.3, 0.7])
        paid = order_total if status == 'Hoàn thành' else np.random.choice([0, order_total])

        for j, (p_code, p_name, p_qty, p_price) in enumerate(lines):
            row = {h: np.nan for h in ORDER_HEADERS}
            row['Mã đặt hàng'] = order_code
            row['Thời gian'] = time_str
            row['Mã khách hàng'] = customer['Mã khách hàng']
            row['Tên khách hàng'] = customer['Tên khách hàng']
            row['Điện thoại'] = customer['Điện thoại']
            row['Người tạo'] = seller['Tên nhân viên (*)']
            row['Kênh bán'] = 'Facebook' if np.random.rand() > 0.5 else 'Zalo'
            row['Mã hàng'] = p_code
            row['Tên hàng'] = p_name
            row['Số lượng'] = p_qty
            row['Đơn giá'] = p_price
            row['Giá bán'] = p_price
            row['Thành tiền'] = p_qty * p_price
            if j == 0:
                row['Khách cần trả'] = order_total
                row['Khách đã trả'] = paid
                row['Tổng tiền hàng'] = order_total
                row['Trạng thái'] = status
                if paid > 0:
                    row[pay_method] = paid
            order_rows.append(row)
        order_id_counter += 1

df_invoices = pd.DataFrame(invoice_rows)
df_orders = pd.DataFrame(order_rows)

print(f"Synthesized {len(df_invoices)} invoice rows and {len(df_orders)} order rows.")

Synthesized 1710 invoice rows and 578 order rows.


## 3. Export to Files (with Auto-Chunking)

In [4]:
def export_chunked(df, base_path, chunk_size, sheet_name):
    if len(df) <= chunk_size:
        df.to_excel(f"{base_path}.xlsx", index=False, sheet_name=sheet_name)
        print(f"Exported {base_path}.xlsx")
    else:
        num_chunks = len(df) // chunk_size + 1
        for i in range(num_chunks):
            chunk = df.iloc[i*chunk_size : (i+1)*chunk_size]
            if len(chunk) > 0:
                pth = f"{base_path}_part{i+1}.xlsx"
                chunk.to_excel(pth, index=False, sheet_name=sheet_name)
                print(f"Exported {pth} ({len(chunk)} rows)")

export_chunked(df_invoices, '../data/out/excel/FACT_INVOICES', CONFIG['chunk_size'], 'InvoiceTemplate')
export_chunked(df_orders, '../data/out/excel/FACT_ORDERS', CONFIG['chunk_size'], 'OrderTemplate')

df_invoices.to_csv('../data/out/csv/fact_invoices_flat.csv', index=False)
df_orders.to_csv('../data/out/csv/fact_orders_flat.csv', index=False)

print("FACT data (Invoices + Orders) exported successfully.")

Exported ../data/out/excel/FACT_INVOICES_part1.xlsx (1000 rows)


Exported ../data/out/excel/FACT_INVOICES_part2.xlsx (710 rows)


Exported ../data/out/excel/FACT_ORDERS.xlsx
FACT data (Invoices + Orders) exported successfully.
